# AXE 2 — 01. Build Interactions
_Notebook 1/2 — Unification Spotify + Netflix + YouTube → `warehouse/interactions.parquet`_

## Approche : Virtual Users (mois)
Avec un seul utilisateur réel, ALS PySpark échoue (matrice rang-1 non-définie).  
**Solution** : chaque mois d'activité = un virtual user.

- `user_id` = `year * 100 + month` (ex: 202401 = Janvier 2024)
- ~80 virtual users → ALS peut faire de vraie collaborative filtering
- Items co-consommés dans le même mois → similaires dans l'espace latent

**Output** : `warehouse/interactions.parquet`  
Colonnes : `user_id`, `item_id`, `item_title`, `platform`, `play_count`

In [ ]:
# ── 0. SETUP ──────────────────────────────────────────────────────────────────
# Si ce notebook plante avec "Connection refused" ou "no resources" :
#   → Kernel > Restart Kernel and Clear Outputs, puis relancer depuis ici.

import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType

spark = SparkSession.builder \
    .appName("MyDigitalTwin-ALS-Interactions") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")
print(f"App : {spark.sparkContext.appName}")

WAREHOUSE = "/opt/spark/warehouse" if os.path.exists("/opt/spark/warehouse") \
            else "C:/Users/arnau/Documents/MyDigitalTwin/warehouse"

print(f"Warehouse: {WAREHOUSE}")
assert os.path.exists(WAREHOUSE), f"Warehouse introuvable: {WAREHOUSE}"

def read_table(name):
    return spark.read.parquet(os.path.join(WAREHOUSE, name))

In [2]:
# ── 1. SPOTIFY STREAMS ────────────────────────────────────────────────────────
# Virtual user = year * 100 + month
# listen_year et listen_month sont déjà dans le parquet

spotify_raw = read_table("spotify_streams")
print(f"Spotify brut : {spotify_raw.count():,} streams")
spotify_raw.printSchema()

spotify = spotify_raw.select(
    F.concat_ws(" — ", F.col("artistName"), F.col("trackName")).alias("item_title"),
    F.lit("spotify").alias("platform"),
    # user_id = year * 100 + month (ex: 202401)
    (F.col("listen_year") * 100 + F.month(F.col("listen_ts"))).cast(IntegerType()).alias("user_id")
).filter(
    F.col("item_title").isNotNull() &
    (F.trim(F.col("item_title")) != "") &
    (F.col("item_title") != " — ") &
    F.col("user_id").isNotNull()
)

print(f"Spotify filtré : {spotify.count():,}")

Spotify brut : 33,972 streams
root
 |-- artistName: string (nullable = true)
 |-- trackName: string (nullable = true)
 |-- msPlayed: long (nullable = true)
 |-- minutes_played: double (nullable = true)
 |-- listen_ts: timestamp (nullable = true)
 |-- listen_year: integer (nullable = true)
 |-- listen_month: string (nullable = true)
 |-- listen_hour: integer (nullable = true)
 |-- listen_weekday: integer (nullable = true)
 |-- listen_week: integer (nullable = true)
 |-- is_night: boolean (nullable = true)
 |-- interaction_weight: double (nullable = true)

Spotify filtré : 33,972


In [3]:
# ── 2. NETFLIX VIEWS ──────────────────────────────────────────────────────────
# watch_year disponible ; watch_month est au format 'yyyy-MM' → extraire le numéro

netflix_raw = read_table("netflix_views")
print(f"Netflix brut : {netflix_raw.count():,} views")

netflix = netflix_raw.select(
    F.col("show_title").alias("item_title"),
    F.lit("netflix").alias("platform"),
    (F.col("watch_year") * 100 + F.month(F.col("watch_date"))).cast(IntegerType()).alias("user_id")
).filter(
    F.col("item_title").isNotNull() &
    (F.trim(F.col("item_title")) != "") &
    F.col("user_id").isNotNull()
)

print(f"Netflix filtré : {netflix.count():,}")

Netflix brut : 4,288 views
Netflix filtré : 4,240


In [ ]:
# ── 3. YOUTUBE — EXCLU ────────────────────────────────────────────────────────
# YouTube retiré des interactions : trop de bruit (pubs, vidéos courtes, auto-play)
# et les titres YouTube apportent peu de valeur sémantique pour la recommandation.
# Sources retenues : Spotify (signal fort msPlayed) + Netflix (contenu long format).
print("YouTube exclu des interactions (trop de bruit).")

In [ ]:
# ── 4. UNION + AGRÉGATION ─────────────────────────────────────────────────────
# Sources : Spotify + Netflix uniquement
raw_all = spotify.union(netflix)
print(f"Total événements : {raw_all.count():,}")

interactions_agg = raw_all.groupBy("user_id", "item_title", "platform").agg(
    F.count("*").alias("play_count")
)

n_users    = interactions_agg.select("user_id").distinct().count()
n_items_raw = interactions_agg.select("item_title").distinct().count()
print(f"Virtual users (mois) : {n_users}")
print(f"Items distincts (brut) : {n_items_raw:,}")

interactions_agg.groupBy("platform").agg(
    F.countDistinct("item_title").alias("n_items"),
    F.countDistinct("user_id").alias("n_months")
).orderBy("platform").show()

In [6]:
# ── 5. FILTRE BRUIT ───────────────────────────────────────────────────────────
# Exclure les items vus dans < 2 mois distincts (bruit)
# Un item vu plusieurs mois = signal plus fiable

item_month_count = interactions_agg.groupBy("item_title").agg(
    F.countDistinct("user_id").alias("n_months_seen")
).filter(F.col("n_months_seen") >= 2)

interactions_filtered = interactions_agg.join(item_month_count.select("item_title"), on="item_title", how="inner")

n_filtered = interactions_filtered.select("item_title").distinct().count()
print(f"Items après filtre (vus dans >= 2 mois) : {n_filtered:,}")
print(f"Interactions : {interactions_filtered.count():,}")

Items après filtre (vus dans >= 2 mois) : 4,580


Interactions : 17,340


In [7]:
# ── 6. STRING INDEXER → item_id entier ────────────────────────────────────────
# ALS nécessite des IDs entiers

from pyspark.ml.feature import StringIndexer

indexer = StringIndexer(inputCol="item_title", outputCol="item_id_float", handleInvalid="keep")
indexer_model = indexer.fit(interactions_filtered)
interactions_indexed = indexer_model.transform(interactions_filtered) \
    .withColumn("item_id", F.col("item_id_float").cast(IntegerType())) \
    .drop("item_id_float")

n_items = interactions_indexed.select("item_id").distinct().count()
print(f"Items indexés : {n_items:,}")
interactions_indexed.orderBy(F.desc("play_count")).show(10, truncate=50)

Items indexés : 4,580


+----------------------+-------+--------+----------+-------+
|            item_title|user_id|platform|play_count|item_id|
+----------------------+-------+--------+----------+-------+
|      Naruto Shippuden| 202303| netflix|       166|    274|
|Hunter X Hunter (2011)| 202501| netflix|       146|   2254|
|      Naruto Shippuden| 202302| netflix|       142|    274|
|                Naruto| 202302| netflix|       139|   2510|
| The Seven Deadly Sins| 202505| netflix|        88|   2722|
|          Regular Show| 202402| netflix|        78|   4150|
|            Fairy Tail| 202007| netflix|        76|   1481|
|            Fairy Tail| 202006| netflix|        65|   1481|
|      Naruto Shippuden| 201909| netflix|        65|    274|
|                Naruto| 201906| netflix|        60|   2510|
+----------------------+-------+--------+----------+-------+
only showing top 10 rows



In [8]:
# ── 7. ÉCRITURE warehouse/interactions ────────────────────────────────────────

out_df = interactions_indexed.select(
    "user_id", "item_id", "item_title", "platform", "play_count"
)

out_path = os.path.join(WAREHOUSE, "interactions")
out_df.write.mode("overwrite").parquet(out_path)

print(f"Écrit : {out_path}")

# Vérification
check = spark.read.parquet(out_path)
print(f"Lignes : {check.count():,}")
check.printSchema()
check.orderBy(F.desc("play_count")).show(10, truncate=50)

spark.stop()
print("Notebook 01 terminé. Lance 02_als_model.ipynb.")

Écrit : /opt/spark/warehouse/interactions
Lignes : 17,340
root
 |-- user_id: integer (nullable = true)
 |-- item_id: integer (nullable = true)
 |-- item_title: string (nullable = true)
 |-- platform: string (nullable = true)
 |-- play_count: long (nullable = true)

+-------+-------+----------------------+--------+----------+
|user_id|item_id|            item_title|platform|play_count|
+-------+-------+----------------------+--------+----------+
| 202303|    274|      Naruto Shippuden| netflix|       166|
| 202501|   2254|Hunter X Hunter (2011)| netflix|       146|
| 202302|    274|      Naruto Shippuden| netflix|       142|
| 202302|   2510|                Naruto| netflix|       139|
| 202505|   2722| The Seven Deadly Sins| netflix|        88|
| 202402|   4150|          Regular Show| netflix|        78|
| 202007|   1481|            Fairy Tail| netflix|        76|
| 202006|   1481|            Fairy Tail| netflix|        65|
| 201909|    274|      Naruto Shippuden| netflix|        65|
| 